# PPO Continuous State and Reward Lab

This notebook runs PPO continuous baselines while keeping the editable experiment definition in a few visible cells. The intended edit points are the state-feature list, the reward weights/scales, and the usual training knobs. The training code itself stays in the package and the notebook writes JSON spec files so each run can be repeated later.


## 1. Setup

Run this first. It finds the project root, imports the notebook helpers, and loads the state/reward spec builders used by the subprocess runner.


In [ ]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_literal_env').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            for path_to_add in (root.parent, root):
                if str(path_to_add) not in sys.path:
                    sys.path.insert(0, str(path_to_add))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from notebooks._teleop_nb import load_json, project_python_executable, repo_paths, show_image, show_rows
from TeleopWithRL import config as cfg
from TeleopWithRL.matlab_literal_env.policy_gradient_experiments.paths import suite_root as policy_gradient_suite_root
from TeleopWithRL.matlab_literal_env.studies.common import save_json
from TeleopWithRL.matlab_literal_env.studies.dqn_state_variants import (
    available_custom_state_feature_rows,
    build_custom_dqn_state_variant_from_spec,
)
from TeleopWithRL.matlab_literal_env.studies.rewarding import (
    DEFAULT_ACTION_DELTA_SCALE_V,
    DEFAULT_ACTION_SCALE_V,
    DEFAULT_FORCE_DIFF_SCALE_N,
    DEFAULT_TRACKING_SCALE_M,
    DEFAULT_TRANSPARENCY_SCALE_W,
    DEFAULT_VELOCITY_ERROR_SCALE_MPS,
    compute_reward_terms,
    reward_formula_from_context,
    reward_variant_from_spec,
)

P = repo_paths()
REPO = P['repo']
WORKSPACE = REPO.parent
PYTHON = project_python_executable(REPO)
PG_RESULTS = REPO / 'matlab_literal_env' / 'policy_gradient_experiments' / 'results'

def short_hash(payload: dict) -> str:
    raw = json.dumps(payload, sort_keys=True, default=str).encode('utf-8')
    return hashlib.sha1(raw).hexdigest()[:8]


## 2. Experiment Knobs

These are the normal run settings. For quick smoke tests, reduce `train_episodes`, `parallel_envs`, and `test_episodes`; for serious comparisons, keep the seed fixed and change one idea at a time.


In [ ]:
ALGO_KEY = 'ppo_continuous'
ALGO_LABEL = 'PPO Continuous'
ALGO_TAG = 'ppo'
RUN_DIR = 'ppo'

CFG = {
    'experiment_label': 'state_reward_lab01',
    'env_mode': 'changing_skin_fat',
    'episode_duration_s': 30.0,
    'env_switch_time_s': 10.0,
    'reset_position_mode': 'midpoint',
    'stroke_limit_mode': 'clamp',
    'force_amp_N': 5.0,
    'force_bias_N': 15.0,
    'force_freq_rad_s': 6.0,
    'force_phase_rad': 0.0,
    'force_waveform': 'sine',
    'train_episodes': 2500,
    'parallel_envs': 8,
    'eval_every_episodes': 100,
    'test_episodes': 100,
    'seed': 42,
    'parallel_workers': 1,
    'worker_torch_threads': 1,
    'skip_existing': True,
}

MODE_LABELS = {
    'dyn': 'switched_dynamics',
    'gui': 'gui_skin_locked',
}


## 3. State Space

This is the main state-space cell. The default is a full physical MDP-style state: master/slave positions, velocities, chamber pressures, line mass flows, valve states, forces, previous action, and time/context. Change `True` to `False` to remove a variable, or turn on one of the derived variables near the bottom.


In [ ]:
USE_STATE_FEATURE = {
    # Physical plant state: positions and velocities
    'x_m': True,
    'x_s': True,
    'v_m': True,
    'v_s': True,

    # Physical plant state: chamber pressures
    'P_m1': True,
    'P_m2': True,
    'P_s1': True,
    'P_s2': True,

    # Physical plant state: tube mass-flow states
    'mdot_L1': True,
    'mdot_L2': True,

    # Physical plant state: valve dynamics
    'x_v': True,
    'x_v_dot': True,

    # Exogenous inputs, measured forces, and context
    'F_h': True,
    'F_e': True,
    'u_v': True,
    'env_id': True,
    'time_fraction': True,

    # Optional action/context variables
    'requested_u_v': False,

    # Optional derived variables. These are redundant if the raw variables above are present,
    # but they can help smaller networks by giving common errors directly.
    'tracking_error': False,
    'velocity_error': False,
    'transparency_error': False,
    'force_diff': False,
    'delta_P_m': False,
    'delta_P_s': False,
    'P_m1_minus_P_s1': False,
    'P_m2_minus_P_s2': False,

    # Optional equilibrium-centered positions. Usually choose these OR absolute x_m/x_s.
    'x_m_eq': False,
    'x_s_eq': False,
    'x_m_centered': False,
    'x_s_centered': False,
}

ALL_STATE_VARIABLES = available_custom_state_feature_rows()
state_lookup = {row['feature']: row for row in ALL_STATE_VARIABLES}
unknown_features = [feature for feature in USE_STATE_FEATURE if feature not in state_lookup]
if unknown_features:
    raise KeyError(f'Unknown state feature(s): {unknown_features}')

def state_feature_group(feature):
    if feature in {'x_m', 'x_s', 'x_m_eq', 'x_s_eq', 'x_m_centered', 'x_s_centered', 'tracking_error'}:
        return 'position'
    if feature in {'v_m', 'v_s', 'velocity_error'}:
        return 'velocity'
    if feature in {'P_m1', 'P_m2', 'P_s1', 'P_s2', 'delta_P_m', 'delta_P_s', 'P_m1_minus_P_s1', 'P_m2_minus_P_s2'}:
        return 'pressure'
    if feature in {'mdot_L1', 'mdot_L2'}:
        return 'mass_flow'
    if feature in {'x_v', 'x_v_dot'}:
        return 'valve'
    if feature in {'F_h', 'F_e', 'force_diff', 'transparency_error'}:
        return 'force_transparency'
    if feature in {'u_v', 'requested_u_v'}:
        return 'action_memory'
    if feature in {'env_id', 'time_fraction'}:
        return 'context'
    return 'other'

catalog_rows = []
for row in ALL_STATE_VARIABLES:
    feature = row['feature']
    display_row = dict(row)
    display_row['group'] = state_feature_group(feature)
    display_row['selected'] = bool(USE_STATE_FEATURE.get(feature, False))
    catalog_rows.append(display_row)
show_rows(catalog_rows, title='Full MDP variable catalog', max_rows=80)

SELECTED_STATE_FEATURES = [
    feature for feature, enabled in USE_STATE_FEATURE.items()
    if enabled
]

STATE_SPEC = {
    'name': 'full_mdp_user_selected',
    'description': 'User-selected variables from the full teleoperation MDP catalog.',
    'selected_features': SELECTED_STATE_FEATURES,
}

STATE_VARIANT = build_custom_dqn_state_variant_from_spec(STATE_SPEC)
selected_state_rows = []
for idx, feature in enumerate(STATE_VARIANT.feature_names, start=1):
    row = dict(state_lookup[feature])
    row['order'] = idx
    row['group'] = state_feature_group(feature)
    selected_state_rows.append(row)

show_rows(selected_state_rows, title=f'Selected state space: {STATE_VARIANT.name}', max_rows=80)
print(f'Observation dimension: {STATE_VARIANT.obs_dim}')


## 4. Action Space

PPO continuous keeps the same action interface: one continuous valve-voltage command clipped to the environment voltage range.


In [ ]:
ACTION_SPACE = {
    'action': 'u_v',
    'type': 'continuous voltage',
    'low_v': float(np.min(cfg.V_LEVELS)),
    'high_v': float(np.max(cfg.V_LEVELS)),
    'scale_used_for_state_and_reward': DEFAULT_ACTION_SCALE_V,
}
show_rows([ACTION_SPACE], title='Action space used by PPO continuous', max_rows=5)


## 5. Reward Function

This cell controls the **structure** of the reward. Each row in `REWARD_TERMS` chooses what signal to use, how to shape it, and whether it is a penalty or a bonus. Set a term aside by deleting it or commenting it out; add a new term by copying one row and changing `name`, `source`, and `shape`.


In [ ]:
REWARD_SOURCE_CATALOG = [
    {'source': 'pos_error', 'meaning': 'x_m - x_s tracking error [m]'},
    {'source': 'velocity_error', 'meaning': 'v_m - v_s [m/s]'},
    {'source': 'transparency_error', 'meaning': '(F_e * v_m) - (F_h * v_s) [W]'},
    {'source': 'force_diff', 'meaning': 'F_e - F_h [N]'},
    {'source': 'u_v', 'meaning': 'applied valve voltage [V]'},
    {'source': 'action_delta', 'meaning': 'u_v - previous_u_v [V]'},
    {'source': 'F_h', 'meaning': 'human/master force input [N]'},
    {'source': 'F_e', 'meaning': 'environment/slave force [N]'},
    {'source': 'x_m', 'meaning': 'absolute master position [m]'},
    {'source': 'x_s', 'meaning': 'absolute slave position [m]'},
    {'source': 'v_m', 'meaning': 'master velocity [m/s]'},
    {'source': 'v_s', 'meaning': 'slave velocity [m/s]'},
    {'source': 'P_m1', 'meaning': 'master chamber 1 pressure [Pa]'},
    {'source': 'P_m2', 'meaning': 'master chamber 2 pressure [Pa]'},
    {'source': 'P_s1', 'meaning': 'slave chamber 1 pressure [Pa]'},
    {'source': 'P_s2', 'meaning': 'slave chamber 2 pressure [Pa]'},
    {'source': 'delta_P_m', 'meaning': 'P_m1 - P_m2 [Pa]'},
    {'source': 'delta_P_s', 'meaning': 'P_s1 - P_s2 [Pa]'},
    {'source': 'mdot_L1', 'meaning': 'tube mass-flow state 1 [kg/s]'},
    {'source': 'mdot_L2', 'meaning': 'tube mass-flow state 2 [kg/s]'},
    {'source': 'x_v', 'meaning': 'valve spool position'},
    {'source': 'x_v_dot', 'meaning': 'valve spool velocity'},
    {'source': 'edge_severity', 'meaning': '0 away from stroke edge, 1 at edge; uses edge_buffer_m'},
    {'source': 'time_fraction', 'meaning': 'episode progress from 0 to 1'},
    {'source': 'env_id', 'meaning': 'skin=0, fat=1'},
]

REWARD_SHAPES = [
    {'shape': 'square', 'formula': '((source - target) / scale)^2'},
    {'shape': 'absolute', 'formula': 'abs((source - target) / scale)'},
    {'shape': 'deadband_square', 'formula': 'max(abs(source - target) - deadband, 0)^2 / scale^2'},
    {'shape': 'deadband_abs', 'formula': 'max(abs(source - target) - deadband, 0) / scale'},
    {'shape': 'above_threshold_square', 'formula': 'max(source - threshold, 0)^2 / scale^2'},
    {'shape': 'below_threshold_square', 'formula': 'max(threshold - source, 0)^2 / scale^2'},
    {'shape': 'tolerance_bonus', 'formula': 'max(0, 1 - abs(source - target) / margin)'},
    {'shape': 'gaussian_bonus', 'formula': 'exp(-0.5 * ((source - target) / scale)^2)'},
]

show_rows(REWARD_SOURCE_CATALOG, title='Reward sources you can use', max_rows=80)
show_rows(REWARD_SHAPES, title='Reward shapes you can use', max_rows=20)

REWARD_TERMS = [
    {
        'name': 'tracking_deadband',
        'source': 'pos_error',
        'shape': 'deadband_square',
        'sign': 'penalty',
        'weight': 40.0,
        'scale': float(cfg.MAX_POSITION_ERROR),
        'deadband': 0.002,
    },
    {
        'name': 'transparency_abs',
        'source': 'transparency_error',
        'shape': 'absolute',
        'sign': 'penalty',
        'weight': 1.5,
        'scale': float(cfg.MAX_POWER_ERROR),
    },
    {
        'name': 'smooth_action',
        'source': 'action_delta',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 0.05,
        'scale': DEFAULT_ACTION_DELTA_SCALE_V,
    },
    {
        'name': 'effort',
        'source': 'u_v',
        'shape': 'square',
        'sign': 'penalty',
        'weight': 0.01,
        'scale': DEFAULT_ACTION_SCALE_V,
    },
    # Example bonus: uncomment to reward very small tracking error directly.
    # {
    #     'name': 'near_zero_tracking_bonus',
    #     'source': 'pos_error',
    #     'shape': 'tolerance_bonus',
    #     'sign': 'bonus',
    #     'weight': 0.5,
    #     'target': 0.0,
    #     'margin': 0.005,
    # },
    # Example pressure regularizer: uncomment to discourage large master pressure imbalance.
    # {
    #     'name': 'master_pressure_balance',
    #     'source': 'delta_P_m',
    #     'shape': 'absolute',
    #     'sign': 'penalty',
    #     'weight': 0.1,
    #     'scale': float(cfg.OBS_SCALE_PRESSURE),
    # },
]

REWARD_SPEC = {
    'name': 'structured_tracking_transparency_smooth',
    'description': 'Notebook-defined structured reward: tracking deadband, absolute transparency, smoothness, and effort.',
    'terms': REWARD_TERMS,
    'penalties': {
        'stroke_limit': 250.0,
        'invalid_state': 100.0,
        'tracking_error_fail': 1000.0,
        'edge_buffer_m': 0.0,
        'low_force_threshold_n': 0.0,
    },
}

REWARD_VARIANT = reward_variant_from_spec(REWARD_SPEC)
reward_rows = []
for term in REWARD_VARIANT.formula_terms:
    reward_rows.append({
        'name': term['name'],
        'source': term['source'],
        'shape': term['shape'],
        'sign': term['sign'],
        'weight': term['weight'],
        'scale': term['scale'],
        'target': term.get('target', 0.0),
        'deadband': term.get('deadband', 0.0),
        'threshold': term.get('threshold', 0.0),
        'margin': term.get('margin', ''),
    })
show_rows(reward_rows, title=f'Reward formula: {REWARD_VARIANT.name}', max_rows=40)


## 6. Reward Sanity Check

This cell evaluates the reward formula on hand-made scenarios before training. If a bad scenario scores better than a good one, fix the reward before launching PPO.


In [ ]:
def reward_context(pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    f_h = 10.0
    f_e = f_h + force_diff_n
    v_s = 0.0
    v_m = velocity_error_mps
    x_s = 0.5 * float(cfg.L_CYL)
    x_m = x_s + pos_error_m
    context = {
        'time': 0.0,
        'time_fraction': 0.0,
        'env_id': 0.0,
        'x_m': x_m,
        'x_s': x_s,
        'x_m_centered': pos_error_m,
        'x_s_centered': 0.0,
        'v_m': v_m,
        'v_s': v_s,
        'P_m1': float(cfg.P_SUPPLY),
        'P_m2': float(cfg.P_SUPPLY),
        'P_s1': float(cfg.P_SUPPLY),
        'P_s2': float(cfg.P_SUPPLY),
        'delta_P_m': 0.0,
        'delta_P_s': 0.0,
        'P_m1_minus_P_s1': 0.0,
        'P_m2_minus_P_s2': 0.0,
        'mdot_L1': 0.0,
        'mdot_L2': 0.0,
        'x_v': 0.0,
        'x_v_dot': 0.0,
        'F_h': f_h,
        'F_e': f_e,
        'u_v': u_v,
        'requested_u_v': u_v,
        'action_delta': action_delta_v,
        'pos_error': pos_error_m,
        'tracking_error': pos_error_m,
        'velocity_error': velocity_error_mps,
        'transparency_error': transparency_error_w,
        'force_diff': force_diff_n,
        'edge_severity': 0.0,
        'low_force_edge_severity': 0.0,
    }
    for key, value in list(context.items()):
        context[f'abs_{key}'] = abs(float(value))
    return context

def reward_case(name, pos_error_m, transparency_error_w, velocity_error_mps=0.0, force_diff_n=0.0, u_v=0.0, action_delta_v=0.0):
    context = reward_context(
        pos_error_m,
        transparency_error_w,
        velocity_error_mps=velocity_error_mps,
        force_diff_n=force_diff_n,
        u_v=u_v,
        action_delta_v=action_delta_v,
    )
    if REWARD_VARIANT.formula_terms:
        reward, grouped_terms, custom_terms = reward_formula_from_context(context, REWARD_VARIANT)
    else:
        reward, track, transp, effort, jerk, velocity, force_diff = compute_reward_terms(
            pos_error=pos_error_m,
            velocity_error=velocity_error_mps,
            transparency_error=transparency_error_w,
            force_diff=force_diff_n,
            u_v=u_v,
            action_delta=action_delta_v,
            variant=REWARD_VARIANT,
        )
        grouped_terms = {
            'track': track,
            'transparency': transp,
            'effort': effort,
            'jerk': jerk,
            'velocity': velocity,
            'force_diff': force_diff,
        }
        custom_terms = {}
    row = {
        'scenario': name,
        'reward': round(float(reward), 4),
    }
    for key, value in grouped_terms.items():
        if abs(float(value)) > 1e-12:
            row[key] = round(float(value), 4)
    for key, value in custom_terms.items():
        row[f'term_{key}'] = round(float(value), 4)
    return row

sanity_rows = [
    reward_case('good tracking + good transparency', 0.002, 1.0, u_v=0.5),
    reward_case('good tracking + poor transparency', 0.002, 15.0, u_v=0.5),
    reward_case('poor tracking + good transparency', 0.040, 1.0, u_v=0.5),
    reward_case('large action change', 0.002, 1.0, u_v=5.0, action_delta_v=5.0),
]
show_rows(sanity_rows, title='Reward sanity check', max_rows=10)


## 7. Build Run Command

This writes the state and reward specs to JSON, builds the command, and shows the exact run configuration. The specs are passed to the subprocess, so the notebook edits are actually used during training.


In [ ]:
RUN_SPEC = {
    'cfg': CFG,
    'state': STATE_SPEC,
    'reward': REWARD_SPEC,
}
SPEC_HASH = short_hash(RUN_SPEC)
CFG['study_name'] = f"{ALGO_TAG}_{CFG['experiment_label']}_{SPEC_HASH}"

SPEC_DIR = PG_RESULTS / 'specs'
STATE_SPEC_PATH = SPEC_DIR / f"{CFG['study_name']}_state.json"
REWARD_SPEC_PATH = SPEC_DIR / f"{CFG['study_name']}_reward.json"
save_json(STATE_SPEC_PATH, STATE_SPEC)
save_json(REWARD_SPEC_PATH, REWARD_SPEC)

RUN_ROOTS = {
    mode: policy_gradient_suite_root(MODE_LABELS[mode], f"{CFG['study_name']}_{mode}") / '00b' / RUN_DIR
    for mode in MODE_LABELS
}

CMD = [
    str(PYTHON),
    '-m',
    'TeleopWithRL.matlab_literal_env.policy_gradient_experiments.run_policy_gradient_baselines_both_fe',
    '--algo', ALGO_KEY,
    '--study-name', CFG['study_name'],
    '--env-mode', CFG['env_mode'],
    '--episode-duration', str(CFG['episode_duration_s']),
    '--env-switch-time', str(CFG['env_switch_time_s']),
    '--reset-position-mode', CFG['reset_position_mode'],
    '--stroke-limit-mode', CFG['stroke_limit_mode'],
    '--force-amp', str(CFG['force_amp_N']),
    '--force-bias', str(CFG['force_bias_N']),
    '--force-freq-rad', str(CFG['force_freq_rad_s']),
    '--force-phase', str(CFG['force_phase_rad']),
    '--force-waveform', CFG['force_waveform'],
    '--reward-variant', REWARD_VARIANT.name,
    '--state-variant', STATE_VARIANT.name,
    '--reward-spec-json', str(REWARD_SPEC_PATH),
    '--state-spec-json', str(STATE_SPEC_PATH),
    '--train-episodes', str(CFG['train_episodes']),
    '--parallel-envs', str(CFG['parallel_envs']),
    '--eval-every-episodes', str(CFG['eval_every_episodes']),
    '--test-episodes', str(CFG['test_episodes']),
    '--seed', str(CFG['seed']),
    '--parallel-workers', str(CFG['parallel_workers']),
    '--worker-torch-threads', str(CFG['worker_torch_threads']),
]
if CFG['skip_existing']:
    CMD.append('--skip-existing')

show_rows(
    [{
        'algo': ALGO_LABEL,
        'study_name': CFG['study_name'],
        'spec_hash': SPEC_HASH,
        'reward_variant': REWARD_VARIANT.name,
        'state_variant': STATE_VARIANT.name,
        'state_features': ', '.join(STATE_VARIANT.feature_names),
        'episode_duration_s': CFG['episode_duration_s'],
        'env_switch_time_s': CFG['env_switch_time_s'],
        'stroke_limit_mode': CFG['stroke_limit_mode'],
        'force_amp_N': CFG['force_amp_N'],
        'force_bias_N': CFG['force_bias_N'],
        'force_freq_rad_s': CFG['force_freq_rad_s'],
        'train_episodes': CFG['train_episodes'],
        'parallel_envs': CFG['parallel_envs'],
        'test_episodes': CFG['test_episodes'],
        'python_executable': str(PYTHON),
        'state_spec_json': str(STATE_SPEC_PATH),
        'reward_spec_json': str(REWARD_SPEC_PATH),
        'dyn_run_root': str(RUN_ROOTS['dyn']),
        'gui_run_root': str(RUN_ROOTS['gui']),
        'command': subprocess.list2cmdline(CMD),
    }],
    title=f'{ALGO_LABEL} run config',
    max_rows=10,
)


## 8. Run Training

This launches both FE modes: switched dynamics and GUI skin-locked evaluation. It can take a while for full `train_episodes`.


In [ ]:
print(subprocess.list2cmdline(CMD))
completed = subprocess.run(CMD, cwd=str(WORKSPACE), check=True)
print(f'Completed with return code {completed.returncode}.')


## 9. Numeric Summary

Run after training. Missing rows usually mean the command has not been run yet, or `skip_existing` pointed at an old incomplete run folder.


In [ ]:
summary_rows = []
metric_rows = []
artifact_rows = []
for mode, run_root in RUN_ROOTS.items():
    summary_path = run_root / 'l' / 'summary.json'
    plots_dir = run_root / 'p'
    artifact_rows.append({
        'fe_mode': MODE_LABELS[mode],
        'summary_json': str(summary_path),
        'plots_dir': str(plots_dir),
        'state_spec_json': str(STATE_SPEC_PATH),
        'reward_spec_json': str(REWARD_SPEC_PATH),
    })
    if not summary_path.exists():
        summary_rows.append({'fe_mode': MODE_LABELS[mode], 'status': 'missing', 'run_root': str(run_root)})
        continue
    data = load_json(summary_path)
    reset_options = dict(data.get('reset_options', {}))
    reward_config = dict(data.get('reward_config', {}))
    state_features = data.get('state_features') or []
    summary_rows.append({
        'fe_mode': MODE_LABELS[mode],
        'algo': data.get('algo_display_name', data.get('algo')),
        'label': data.get('label'),
        'tracking_rmse_m': data.get('tracking_rmse_m'),
        'transparency_rmse_w': data.get('transparency_rmse_w'),
        'pre_switch_tracking_rmse_m': data.get('pre_switch_tracking_rmse_m'),
        'post_switch_tracking_rmse_m': data.get('post_switch_tracking_rmse_m'),
        'pre_switch_transparency_rmse_w': data.get('pre_switch_transparency_rmse_w'),
        'post_switch_transparency_rmse_w': data.get('post_switch_transparency_rmse_w'),
        'invalid_episode_rate': data.get('invalid_episode_rate'),
        'mean_reward': data.get('mean_reward'),
        'episode_duration': data.get('episode_duration'),
        'env_switch_time': data.get('env_switch_time'),
        'force_amp': reset_options.get('force_amp'),
        'force_bias': reset_options.get('force_bias'),
        'force_freq_rad': reset_options.get('force_freq_rad'),
        'reset_position_mode': reset_options.get('reset_position_mode'),
        'state_variant': data.get('state_variant'),
        'state_features': ', '.join(state_features) if isinstance(state_features, list) else state_features,
        'reward_variant': data.get('reward_variant'),
        'reward_terms': len(reward_config.get('formula_terms') or []),
        'reward_structure': ', '.join(term.get('name', '') for term in (reward_config.get('formula_terms') or [])),
        'out_dir': data.get('out_dir'),
    })
    metric_rows.append({
        'fe_mode': MODE_LABELS[mode],
        'track_rmse_mm': round(1000.0 * float(data.get('tracking_rmse_m') or 0.0), 3),
        'transp_rmse_w': round(float(data.get('transparency_rmse_w') or 0.0), 3),
        'pre_track_rmse_mm': round(1000.0 * float(data.get('pre_switch_tracking_rmse_m') or 0.0), 3),
        'post_track_rmse_mm': round(1000.0 * float(data.get('post_switch_tracking_rmse_m') or 0.0), 3),
        'pre_transp_rmse_w': round(float(data.get('pre_switch_transparency_rmse_w') or 0.0), 3),
        'post_transp_rmse_w': round(float(data.get('post_switch_transparency_rmse_w') or 0.0), 3),
        'mean_reward': round(float(data.get('mean_reward') or 0.0), 3),
    })

show_rows(metric_rows, title=f'{ALGO_LABEL} rollout RMSE metrics', max_rows=10)
show_rows(summary_rows, title=f'{ALGO_LABEL} run summaries', max_rows=10)
show_rows(artifact_rows, title=f'{ALGO_LABEL} artifact locations', max_rows=10)


## 10. Reward Component Plot

This reads the saved evaluation rollout and plots reward components separately, so tracking and transparency are no longer hidden inside one scalar.


In [ ]:
import matplotlib.pyplot as plt

FIXED_REWARD_COMPONENTS = [
    'reward_track',
    'reward_transparency',
    'reward_velocity',
    'reward_force_diff',
    'reward_effort',
    'reward_jerk',
    'reward_edge',
    'reward_low_force_edge',
    'reward_terminal_penalty',
]

def plot_reward_components(run_root: Path, title: str):
    npz_path = run_root / 'e' / 'test.npz'
    if not npz_path.exists():
        print(f'Missing evaluation history: {npz_path}')
        return
    data = np.load(npz_path, allow_pickle=True)
    if 'time' not in data:
        print(f'No time vector in {npz_path}')
        return
    t = np.asarray(data['time'], dtype=float)
    custom_keys = sorted(key for key in data.files if key.startswith('reward_term_'))
    component_keys = custom_keys if custom_keys else [key for key in FIXED_REWARD_COMPONENTS if key in data]
    fig, ax = plt.subplots(figsize=(14, 5))
    plotted = 0
    for key in component_keys:
        values = np.asarray(data[key], dtype=float)
        n = min(t.size, values.size)
        if n == 0:
            continue
        label = key.replace('reward_term_', '').replace('reward_', '')
        ax.plot(t[:n], values[:n], lw=1.2, label=label)
        plotted += 1
    if plotted == 0:
        print(f'No reward component arrays found in {npz_path}')
        return
    ax.axvline(CFG['env_switch_time_s'], color='0.35', ls='--', lw=1.0, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Time [s]')
    ax.set_ylabel('Signed custom contribution' if custom_keys else 'Penalty term contribution')
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=3)
    plt.tight_layout()
    plt.show()

for mode, run_root in RUN_ROOTS.items():
    plot_reward_components(run_root, f"{MODE_LABELS[mode]} reward components")


## 11. Plot Gallery


### switched_dynamics


In [ ]:
plot_specs = [
    ('train.png', 'Training metrics'),
    ('roll.png', 'Evaluation roll plot'),
    ('act.png', 'Evaluation action plot'),
    ('traj.png', 'Evaluation trajectory plot'),
    ('slices.png', 'Policy slices'),
]

run_root = RUN_ROOTS['dyn']
for filename, title in plot_specs:
    show_image(run_root / 'p' / filename, title=f"{MODE_LABELS['dyn']}: {title}")


### gui_skin_locked


In [ ]:
run_root = RUN_ROOTS['gui']
for filename, title in plot_specs:
    show_image(run_root / 'p' / filename, title=f"{MODE_LABELS['gui']}: {title}")
